# Model Results and Analysis

This notebook loads the pre-trained models, evaluates them on the test set, and visualizes their performance metrics and feature importances.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import sys

# Add src directory to Python path to import custom modules
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

# Ensure plots are displayed inline
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Setup and Data Loading

We need to load the test set using our preprocessing pipeline and the saved models.

In [ ]:
# Import the preprocessing function
try:
    from src.data_processing import preprocess_data
    from src.evaluate import get_feature_importances # For feature importance extraction
    from src.models import get_logistic_regression, get_decision_tree, get_random_forest # For dummy model creation
except ModuleNotFoundError as e:
    print(f"Error importing module: {e}. Ensure you are in the 'notebooks' directory or 'src' is in PYTHONPATH.")
    # Define dummy functions if import fails, to allow notebook to open
    def preprocess_data(df_path, **kwargs):
        print("Error: preprocess_data could not be imported. Returning dummy data.")
        # Create minimal dummy data structure that downstream cells might expect
        n_samples = 100
        n_features = 5
        X = pd.DataFrame(np.random.rand(n_samples, n_features), columns=[f'feature_{i}' for i in range(n_features)])
        y = pd.Series(np.random.choice([0,1], size=n_samples))
        return X, X, y, y # X_train, X_test, y_train, y_test
    def get_feature_importances(model, feature_names, top_n=10):
        print("Error: get_feature_importances could not be imported.")
        return pd.DataFrame()
    def get_logistic_regression(**kwargs): return None
    def get_decision_tree(**kwargs): return None
    def get_random_forest(**kwargs): return None

# Define paths
DATA_PATH = '../data/student_data.csv'
MODELS_DIR = '../models/'
REPORTS_DIR = '../reports/figures/'

# Ensure necessary directories exist
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

# Create dummy student_data.csv if it doesn't exist (similar to data_processing.py)
if not os.path.exists(DATA_PATH):
    print(f"'{DATA_PATH}' not found. Creating a dummy CSV for demonstration.")
    dummy_df_data = {
        'G3': np.random.randint(0, 21, size=100), 
        'age': np.random.randint(15,20, size=100),
        'absences': np.random.randint(0,20, size=100),
        'studytime': np.random.randint(1,5, size=100),
        'sex': np.random.choice(['F','M'], size=100)
    }
    for i in range(5): dummy_df_data[f'cat_col_{i}'] = np.random.choice(['A','B','C'], size=100)
    for i in range(3): dummy_df_data[f'num_col_{i}'] = np.random.rand(100) * 10
    pd.DataFrame(dummy_df_data).to_csv(DATA_PATH, index=False)

# Load test data
print("Loading and preprocessing data for test set...")
try:
    X_train, X_test, y_train, y_test = preprocess_data(df_path=DATA_PATH, test_size=0.25, random_state=42)
    print("Test data loaded successfully.")
    print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")
except Exception as e:
    print(f"Error during data preprocessing: {e}")
    # Create placeholder X_test, y_test if preprocessing fails
    X_test = pd.DataFrame({'dummy_feature': np.random.rand(25)})
    y_test = pd.Series(np.random.choice([0,1], size=25))

## 2. Load Trained Models

Load the serialized models saved by `train.py`.

In [ ]:
model_files = {
    "Logistic Regression": os.path.join(MODELS_DIR, "best_logistic_regression.pkl"),
    "Decision Tree": os.path.join(MODELS_DIR, "best_decision_tree.pkl"),
    "Random Forest": os.path.join(MODELS_DIR, "best_random_forest.pkl")
}

loaded_models = {}

# Create dummy fitted models if actual files are missing (for notebook to run)
for model_name, model_path in model_files.items():
    if not os.path.exists(model_path):
        print(f"Warning: Model file {model_path} not found. Creating a dummy fitted model for {model_name}.")
        dummy_model = None
        try:
            if model_name == "Logistic Regression" and get_logistic_regression:
                dummy_model = get_logistic_regression(random_state=42).fit(X_test, y_test)
            elif model_name == "Decision Tree" and get_decision_tree:
                dummy_model = get_decision_tree(random_state=42).fit(X_test, y_test)
            elif model_name == "Random Forest" and get_random_forest:
                dummy_model = get_random_forest(random_state=42, n_estimators=10).fit(X_test, y_test)
            
            if dummy_model:
                joblib.dump(dummy_model, model_path)
                print(f"Dummy model for {model_name} created and saved to {model_path}.")
            else:
                print(f"Could not create dummy model for {model_name} due to missing model function.")
        except Exception as e:
            print(f"Error creating dummy model for {model_name}: {e}")

for model_name, path in model_files.items():
    try:
        if os.path.exists(path):
            loaded_models[model_name] = joblib.load(path)
            print(f"Successfully loaded {model_name} from {path}")
        else:
            print(f"Model file {path} still not found after attempting dummy creation. Skipping {model_name}.")
            loaded_models[model_name] = None # Placeholder
    except Exception as e:
        print(f"Error loading model {model_name} from {path}: {e}")
        loaded_models[model_name] = None

## 3. Reproduce Evaluation Metrics

Display the evaluation metrics DataFrame (`reports/figures/metrics_comparison.csv`). If the file doesn't exist, generate metrics directly.

In [ ]:
metrics_file_path = os.path.join(REPORTS_DIR, 'metrics_comparison.csv')
metrics_df = None

if os.path.exists(metrics_file_path):
    print(f"Loading metrics from {metrics_file_path}")
    metrics_df = pd.read_csv(metrics_file_path)
else:
    print(f"Warning: {metrics_file_path} not found. Generating metrics directly.")
    # Regenerate metrics if file is missing (similar to evaluate.py)
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
    all_metrics_notebook = []
    for name, model in loaded_models.items():
        if model is None:
            print(f"Skipping metrics for {name} as model is not loaded.")
            all_metrics_notebook.append({'model': name, 'accuracy': np.nan, 'precision': np.nan, 'recall': np.nan, 'f1_score': np.nan, 'roc_auc': np.nan})
            continue
        try:
            y_pred = model.predict(X_test)
            y_pred_proba = model.predict_proba(X_test)[:, 1]
            metrics = {
                'model': name,
                'accuracy': accuracy_score(y_test, y_pred),
                'precision': precision_score(y_test, y_pred, zero_division=0),
                'recall': recall_score(y_test, y_pred, zero_division=0),
                'f1_score': f1_score(y_test, y_pred, zero_division=0),
                'roc_auc': roc_auc_score(y_test, y_pred_proba)
            }
            all_metrics_notebook.append(metrics)
        except Exception as e:
            print(f"Error generating metrics for {name}: {e}")
            all_metrics_notebook.append({'model': name, 'accuracy': np.nan, 'precision': np.nan, 'recall': np.nan, 'f1_score': np.nan, 'roc_auc': np.nan})
            
    if all_metrics_notebook:
        metrics_df = pd.DataFrame(all_metrics_notebook)
        metrics_df.to_csv(metrics_file_path, index=False) # Save it for next time
        print(f"Metrics generated and saved to {metrics_file_path}")
    else:
        print("No models available to generate metrics.")

if metrics_df is not None:
    print("\nModel Performance Metrics:")
    display(metrics_df)
else:
    print("\nMetrics DataFrame could not be loaded or generated.")

## 4. Plot Metrics Comparison

In [ ]:
if metrics_df is not None and not metrics_df.empty:
    metrics_to_plot = ['accuracy', 'f1_score', 'roc_auc']
    plot_df = metrics_df.set_index('model')[metrics_to_plot]
    
    plot_df.plot(kind='bar', figsize=(12, 7), rot=0)
    plt.title('Model Comparison: Accuracy, F1-score, ROC AUC', fontsize=16)
    plt.ylabel('Score')
    plt.ylim(0, 1.05)
    plt.legend(title='Metric')
    # Save the figure
    # fig_path = os.path.join(REPORTS_DIR, 'model_metrics_comparison_bar_chart.png')
    # plt.savefig(fig_path)
    # print(f"Saved metrics comparison chart to {fig_path}")
    plt.show()
else:
    print("Metrics DataFrame is not available for plotting.")

## 5. Plot Feature Importances / Coefficients

### 5.1 Decision Tree and Random Forest Feature Importances

In [ ]:
def plot_feature_importances(model, feature_names, model_name_str, top_n=10):
    if model is None or not hasattr(model, 'feature_importances_'):
        print(f"Cannot plot feature importances for {model_name_str} (model not loaded or no importances attribute).")
        return
    
    # Use the get_feature_importances function from evaluate.py (or its dummy)
    importances_df = get_feature_importances(model, feature_names, top_n=top_n)
    
    if not importances_df.empty:
        plt.figure(figsize=(10, top_n * 0.5))
        sns.barplot(x='importance', y='feature', data=importances_df, palette='viridis')
        plt.title(f'Top {top_n} Feature Importances for {model_name_str}', fontsize=14)
        plt.xlabel('Importance')
        plt.ylabel('Feature')
        plt.tight_layout()
        # fig_path = os.path.join(REPORTS_DIR, f'{model_name_str.lower().replace(" ", "_")}_feature_importances.png')
        # plt.savefig(fig_path)
        # print(f"Saved {model_name_str} feature importances plot to {fig_path}")
        plt.show()
    else:
        print(f"No feature importances to plot for {model_name_str}.")

if 'X_test' in locals() and X_test is not None:
    feature_cols = X_test.columns.tolist()
    if loaded_models.get("Decision Tree"):
        plot_feature_importances(loaded_models["Decision Tree"], feature_cols, "Decision Tree")
    else:
        print("Decision Tree model not loaded, skipping feature importance plot.")
        
    if loaded_models.get("Random Forest"):
        plot_feature_importances(loaded_models["Random Forest"], feature_cols, "Random Forest")
    else:
        print("Random Forest model not loaded, skipping feature importance plot.")
else:
    print("X_test not available, cannot plot feature importances.")

### 5.2 Logistic Regression Coefficients

In [ ]:
def plot_logistic_regression_coefficients(model, feature_names, model_name_str="Logistic Regression", top_n=10):
    if model is None or not hasattr(model, 'coef_'):
        print(f"Cannot plot coefficients for {model_name_str} (model not loaded or no coef_ attribute).")
        return
    
    # Use the get_feature_importances function, it handles coefficients too
    coefficients_df = get_feature_importances(model, feature_names, top_n=top_n)

    if not coefficients_df.empty and 'abs_coefficient' in coefficients_df.columns:
        plt.figure(figsize=(10, top_n * 0.5))
        sns.barplot(x='abs_coefficient', y='feature', data=coefficients_df, palette='coolwarm')
        plt.title(f'Top {top_n} Absolute Coefficients for {model_name_str}', fontsize=14)
        plt.xlabel('Absolute Coefficient Value')
        plt.ylabel('Feature')
        plt.tight_layout()
        # fig_path = os.path.join(REPORTS_DIR, f'{model_name_str.lower().replace(" ", "_")}_coefficients.png')
        # plt.savefig(fig_path)
        # print(f"Saved {model_name_str} coefficients plot to {fig_path}")
        plt.show()
    else:
        print(f"No coefficients to plot for {model_name_str}. Ensure 'abs_coefficient' column exists.")

if 'X_test' in locals() and X_test is not None:
    feature_cols = X_test.columns.tolist()
    if loaded_models.get("Logistic Regression"):
        plot_logistic_regression_coefficients(loaded_models["Logistic Regression"], feature_cols)
    else:
        print("Logistic Regression model not loaded, skipping coefficient plot.")
else:
    print("X_test not available, cannot plot coefficients.")

## 6. Summary and Conclusions

### Model Performance:
*(This section should be filled in based on the actual results from `metrics_df`)*

For example:
*   The Random Forest model achieved the highest accuracy (e.g., XX.X%) and ROC AUC (e.g., 0.XXX).
*   The Decision Tree model was slightly less performant but offered more interpretable feature importances directly.
*   Logistic Regression provided a baseline performance and its coefficients can indicate feature influence direction (positive/negative if not taking absolute values initially).

### Feature Importance Insights:
*(This section should be filled in based on the actual feature importance plots)*

For example:
*   Across the tree-based models (Decision Tree, Random Forest), features like `G2` (previous grade), `failures` (past failures), and `higher_yes` (desire for higher education) consistently appeared as top predictors.
*   The Logistic Regression model highlighted similar features, with `absences` also showing a notable (negative) impact on the likelihood of passing.
*   Differences in top features between models can occur due to how they handle feature interactions and non-linearities (e.g., Random Forest can capture more complex relationships).

*(Note: The insights above are examples. Actual conclusions will depend on the data and model outputs. The dummy data used for fallbacks will likely result in non-sensical feature importances.)*